In [ ]:
import polars as pl

In [ ]:
df = pl.read_csv("data.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.null_count().to_dicts()

In [ ]:
df = df.drop("")

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Target distribution
plt.pie(df["diagnosis"].value_counts().to_series(1), labels=["Benign", "Malignant"], autopct="%1.1f%%")
plt.show()

In [ ]:
df

In [ ]:
target_dict = {
    "B": 0,
    "M": 1
}

df = df.with_columns(
    pl.col("diagnosis").replace(target_dict).cast(pl.Int32)
)

In [ ]:
sns.heatmap(df.to_pandas().corr(), cmap='coolwarm', fmt='.2f')

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    df.drop("diagnosis"),
    df["diagnosis"],
    test_size=0.2,
    random_state=42,
    stratify=df["diagnosis"]
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def plot_confusion_matrix(cm):
    sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', 
                xticklabels=['Benign', 'Malignant'], 
                yticklabels=['Benign', 'Malignant'])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()


def evaluate_binary_classifier(model_name, y_true, y_pred, y_prob):
    """
    Evaluate the performance of a binary classifier.

    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted binary labels.
    y_prob (array-like): Predicted probabilities for the positive class.

    """
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    TP = cm[1, 1]
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]


    output_df = pl.DataFrame(
        {
            "Model": [model_name],
            "Accuracy": [accuracy],
            "Precision": [precision],
            "Recall": [recall],
            "F1": [f1],
            "TP" : [TP],
            "TN" : [TN],
            "FP" : [FP],
            "FN" : [FN]
        }
    )
    plot_confusion_matrix(cm)

    return output_df

In [ ]:
results_df = pl.DataFrame(
    schema = {
        "Model": pl.String,
        "Accuracy": pl.Float64,
        "Precision": pl.Float64,
        "Recall": pl.Float64,
        "F1": pl.Float64,
        "TP" : pl.Int64,
        "TN" : pl.Int64,
        "FP" : pl.Int64,
        "FN" : pl.Int64,
    }
)

In [ ]:
results_df

In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("Logistic Regression", y_test, y_pred, y_prob))


In [ ]:
# SVM
from sklearn.svm import SVC

pipeline = make_pipeline(
    StandardScaler(),
    SVC(probability=True)
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("SVM", y_test, y_pred, y_prob))

In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

pipeline = make_pipeline(
    DecisionTreeClassifier()
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("Decision Tree", y_test, y_pred, y_prob))

In [ ]:
# Random Forests
from sklearn.ensemble import RandomForestClassifier

pipeline = make_pipeline(
    RandomForestClassifier()
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("Random Forests", y_test, y_pred, y_prob))

In [ ]:
# Gradient Boosting Machines
from sklearn.ensemble import GradientBoostingClassifier

pipeline = make_pipeline(
    GradientBoostingClassifier()
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("Gradient Boosting Machines", y_test, y_pred, y_prob))

In [ ]:
# XGBoost

import xgboost as xgb

pipeline = make_pipeline(
    xgb.XGBClassifier()
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("XGBoost", y_test, y_pred, y_prob))

In [ ]:
# KNearest Neighbors
from sklearn.neighbors import KNeighborsClassifier

pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:, 1]

results_df = results_df.vstack(evaluate_binary_classifier("KNearest Neighbors", y_test, y_pred, y_prob))

In [ ]:
results_df